In [1]:
from dataclasses import dataclass
from typing import Iterator

import torch
from torch import Tensor
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor


@dataclass(slots=True, frozen=True)
class CifarItem:
    image: Tensor # (3, 32, 32)
    label: Tensor # ()


class CifarDataset:
    images: Tensor  # (N, 3, 32, 32)
    labels: Tensor  # (N,)

    def __init__(self, root: str = 'data/'):
        dataset = CIFAR10(root=root, download=True, transform=ToTensor())

        # Preload all data into memory as tensors
        self.images = torch.stack([dataset[i][0] for i in range(len(dataset))])  # (N, 3, 32, 32)
        self.labels = torch.tensor([dataset[i][1] for i in range(len(dataset))])  # (N,)

        # Share memory for multi-process data loading
        self.images.share_memory_()
        self.labels.share_memory_()

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> CifarItem:
        return CifarItem(image=self.images[index], label=self.labels[index])

    def __iter__(self) -> Iterator[CifarItem]:
        for i in range(len(self)):
            yield self[i]

dataset = CifarDataset()
print(f'Number of samples in CifarInMemoryDataset: {len(dataset)}')

Number of samples in CifarInMemoryDataset: 50000


In [2]:
from typing import final

from matplotlib.pylab import f
from pyparsing import C

from apriori.ico.core.operator import IcoOperator
from apriori.ico.core.source import IcoSource
from apriori.ico.utils.data.batcher import IcoBatcher

cifar_items = IcoSource[CifarItem](lambda: iter(dataset), name='CIFAR10 dataset')
batcher = IcoBatcher[CifarItem](batch_size=8)

data_input_flow = cifar_items | batcher
data_input_flow.name = 'CIFAR10 In-Memory Data Input Flow'
data_input_flow.describe()

for item_batch in data_input_flow(None):
    print(f'Number of items in batch: {len(list(item_batch))}')
    break




──────────────────────────────────────── CIFAR10 In-Memory Data Input Flow ────────────────────────────────────────

CIFAR10 In-Memory Data Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
├── CIFAR10 dataset (source)  () → Iterator[CifarItem]
└── batcher(8) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]

Number of items in batch: 8


In [3]:
from abc import ABC, abstractmethod

from torchvision.transforms import functional as F


class CifarTransform(ABC):
    p: float
    factor: float

    def __init__(self, p: float = 1.0, factor: float = 1.0):
        self.p = p
        self.factor = factor

    @abstractmethod
    def _image_transform(self, image: Tensor, factor: float) -> Tensor:
        raise NotImplementedError

    def __call__(self, item: CifarItem) -> CifarItem:
        if torch.rand(1) >= self.p:
            return item

        return CifarItem(
            image=self._image_transform(item.image, self.factor),
            label=item.label
        )

class HorizontalFlip(CifarTransform):
    def _image_transform(self, image: Tensor, factor: float) -> Tensor:
        return F.hflip(image)


class VerticalFlip(CifarTransform):
    def _image_transform(self, image: Tensor, factor: float) -> Tensor:
        return F.vflip(image)


class AdjustBrightness(CifarTransform):
    def _image_transform(self, image: Tensor, factor: float) -> Tensor:
        return F.adjust_brightness(image, factor)


class AdjustContrast(CifarTransform):
    def _image_transform(self, image: Tensor, factor: float) -> Tensor:
        return F.adjust_contrast(image, factor)


item_aug_flow = (
    IcoOperator(HorizontalFlip())
    | IcoOperator(VerticalFlip())
    | IcoOperator(AdjustBrightness(factor=0.2))
    | IcoOperator(AdjustContrast(factor=0.2))
)
item_aug_flow.name = "Item augmentation flow"
item_aug_flow.describe()




───────────────────────────────────────────── Item augmentation flow ──────────────────────────────────────────────

Item augmentation flow (chain)  CifarItem → CifarItem
├── HorizontalFlip | VerticalFlip | AdjustBrightness (chain)  CifarItem → CifarItem
│   ├── HorizontalFlip | VerticalFlip (chain)  CifarItem → CifarItem
│   │   ├── HorizontalFlip (operator)  CifarItem → CifarItem
│   │   └── VerticalFlip (operator)  CifarItem → CifarItem
│   └── AdjustBrightness (operator)  CifarItem → CifarItem
└── AdjustContrast (operator)  CifarItem → CifarItem

In [4]:
from apriori.ico.core.pipeline import IcoPipeline

item_aug_flow = IcoPipeline(
    HorizontalFlip(),
    VerticalFlip(),
    AdjustBrightness(),
    AdjustContrast(),
    name="Item augmentation flow",
)
item_aug_flow.describe()

───────────────────────────────────────────── Item augmentation flow ──────────────────────────────────────────────

Item augmentation flow (pipeline)  CifarItem → CifarItem
├── HorizontalFlip (operator)  CifarItem → CifarItem
├── VerticalFlip (operator)  CifarItem → CifarItem
├── AdjustBrightness (operator)  CifarItem → CifarItem
└── AdjustContrast (operator)  CifarItem → CifarItem

In [5]:

@final
class CifarBatch:
    __slots__ = ('images', 'labels')

    images: Tensor  # (B, 3, 32, 32)
    labels: Tensor  # (B,)

    def __init__(self, items: Iterator[CifarItem]):
        items_list = list(items)
        self.images = torch.stack([item.image for item in items_list])  # (B, 3, 32, 32)
        self.labels = torch.tensor([item.label for item in items_list])  # (B,)


def collate(items: Iterator[CifarItem]) -> CifarBatch:
    return CifarBatch(items)

def share_memory(batch: CifarBatch) -> CifarBatch:
    batch.images.share_memory_()
    batch.labels.share_memory_()
    return batch

def create_augmentation_flow() -> IcoOperator[Iterator[CifarItem], CifarBatch]:
    aug_flow = item_aug_flow.iterate() | IcoOperator(collate) | IcoOperator(share_memory)
    aug_flow.name = "Full Augmentation flow"
    return aug_flow

aug_flow = create_augmentation_flow()
aug_flow.describe()

───────────────────────────────────────────── Full Augmentation flow ──────────────────────────────────────────────

Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
├── iterate(Item augmentation flow) | collate (chain)  Iterator[CifarItem] → CifarBatch
│   ├── iterate(Item augmentation flow) (iterator)  Iterator[CifarItem] → Iterator[CifarItem]
│   │   └── Item augmentation flow (pipeline)  CifarItem → CifarItem
│   │       ├── HorizontalFlip (operator)  CifarItem → CifarItem
│   │       ├── VerticalFlip (operator)  CifarItem → CifarItem
│   │       ├── AdjustBrightness (operator)  CifarItem → CifarItem
│   │       └── AdjustContrast (operator)  CifarItem → CifarItem
│   └── collate (operator)  Iterator[CifarItem] → CifarBatch
└── share_memory (operator)  CifarBatch → CifarBatch

In [6]:
full_data_flow = data_input_flow | aug_flow.iterate()
full_data_flow.name = "Full Data Flow with Augmentation"
full_data_flow.describe()

──────────────────────────────────────── Full Data Flow with Augmentation ─────────────────────────────────────────

Full Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
├── CIFAR10 In-Memory Data Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│   ├── CIFAR10 dataset (source)  () → Iterator[CifarItem]
│   └── batcher(8) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
└── iterate(Full Augmentation flow) (iterator)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
    └── Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
        ├── iterate(Item augmentation flow) | collate (chain)  Iterator[CifarItem] → CifarBatch
        │   ├── iterate(Item augmentation flow) (iterator)  Iterator[CifarItem] → Iterator[CifarItem]
        │   │   └── Item augmentation flow (pipeline)  CifarItem → CifarItem
        │   │       ├── HorizontalFlip (operator)  CifarItem → CifarItem
        │   │       ├── VerticalFlip (operator)  CifarItem → CifarItem
        │   │       ├── AdjustBrightness (operator)  CifarItem → CifarItem
        │   │       └── AdjustContrast (operator)  CifarItem → CifarItem
        │   └── collate (operator)  Iterator[CifarItem] → CifarBatch
        └── share_memory (operator)  CifarBatch → CifarBatch

In [7]:
for batch in full_data_flow(None):
    print(f'Batch images shape: {batch.images.shape}, Batch labels shape: {batch.labels.shape}')
    break

Batch images shape: torch.Size([8, 3, 32, 32]), Batch labels shape: torch.Size([8])


In [8]:
from apriori.ico.core.async_stream import IcoAsyncStream
from apriori.ico.runtime.agent.mp.mp_process import MPProcess

num_workers = 4
workers_pool = [MPProcess(create_augmentation_flow) for _ in range(num_workers)]

async_stream = IcoAsyncStream(workers_pool, name="Async Augmentation Stream")

parallel_data_flow = data_input_flow | async_stream
parallel_data_flow.name = "Parallel Data Flow with Augmentation"
parallel_data_flow.describe()

────────────────────────────────────── Parallel Data Flow with Augmentation ───────────────────────────────────────

Parallel Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
├── CIFAR10 In-Memory Data Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│   ├── CIFAR10 dataset (source)  () → Iterator[CifarItem]
│   └── batcher(8) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
└── Async Augmentation Stream (async_stream)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
    ├── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch
    ├── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch
    ├── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch
    └── mp_process (operator) [inactive]  Iterator[CifarItem] → CifarBatch